# 05 · Réplica en el año 5

**Solo diagnóstico: no modifica la lista ni los candidatos, y ninguna etapa de selección lo lee (FR-030).** Antes de leer cualquier día de Y5 se declaran todas las claves de `[forecast]`; cambiarlas después exige una nueva versión.

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

In [ ]:
from src.config import forecast_keys, get_key

pd.Series({k: get_key(cfg, k) for k in forecast_keys(cfg)}, name="valor a declarar")

Medidas pareadas de Y5 frente a Y4 (`prev`) y frente a Y1 (`y1`), con la regla ABC, `find_cv_cuts` y los umbrales de racha congelados. Se leen junto a las de train (`stability.parquet`).

In [ ]:
from src.stability import run_replication

rep = run_replication(cfg)
st5 = rep["stability_y5"]
st5.set_index(["base_kind", "base", "target", "level"])[
    ["n", "kappa", "stuart_maxwell", "sm_pvalue", "max_shift_pp", "max_shift_class", "keep_pct", "up_pct", "down_pct", "mixed_pct"]
]

In [ ]:
from src.freeze import load_frozen

train_st = load_frozen("stability")
both = pd.concat([train_st[train_st["base_kind"].isin(["prev", "y1"])], st5])
both.pivot_table(index=["level", "base_kind"], columns="target", values="kappa").round(2)

In [ ]:
m5 = rep["matrix_y5"]
pd.crosstab(m5["abc_class"], m5["xyz_class"].fillna("-"), margins=True)

Productos objetivo en Y5: `same_abc`/`same_xyz` frente a Y4 con los cortes elegidos en Y4; `same_as_frozen` compara la celda de Y5 (cortes congelados de train) con la celda congelada.

In [ ]:
r = rep["year5_replication"]
print(f"mantienen su celda congelada: {r['same_as_frozen'].mean():.1%}")
display(r[["same_abc", "same_xyz", "same_cell"]].mean().to_frame("% frente a Y4"))
r[~r["same_as_frozen"]]